In [10]:
import numpy as np
import pandas as pd


In [11]:
df =  pd.DataFrame([[8,8,4],[7,9,5],[6,10,6],[5,12,7]], columns = ['cgpa','profile_score','lpa'])

In [150]:
df

,cgpa,profile_score,lpa
0,8,8,4
1,7,9,5
2,6,10,6
3,5,12,7


In [151]:
def initialize_parameter(layer_dims):
    np.random.seed(3)
    parameters = {}
    L = len(layer_dims)

    for l in range(1,L):
        parameters["w" + str(l)] = np.ones((layer_dims[l-1], layer_dims[l]))*0.1
        parameters["b"+ str(l)] = np.zeros((layer_dims[l],1))
    return parameters

In [152]:
def linear_forward(A_prev,w,b):
    z = np.dot(w.T,A_prev) +b
    return z

In [153]:
df

,cgpa,profile_score,lpa
0,8,8,4
1,7,9,5
2,6,10,6
3,5,12,7


In [158]:
# Function for Forward Prop
def L_layer_forward(X, parameters):
    A = X
    L = len(parameters) // 2
    for l in range(1, L+1):
        A_prev = A
        wl = parameters["w"+str(l)]
        bl = parameters["b"+str(l)]
        #print("A"+str(l-1)+": ",A_prev)
        #print("w"+str(l)+": ",wl)
        #print("b"+str(l)+": ",bl)
        #print("--"*20)

        A = linear_forward(A_prev,wl,bl)
        #print("A"+str(l)+": ",A)
        
    return A,A_prev

In [159]:
def update_parameters(parameters, y, y_hat, A1, X):

    error = (y - y_hat).item()

    parameters['w2'][0][0] += 0.001 * 2 * error * A1[0][0]
    parameters['w2'][1][0] += 0.001 * 2 * error * A1[1][0]

    parameters['b2'][0][0] += 0.001 * 2 * error

    parameters['w1'][0][0] += 0.001 * 2 * error * parameters['w2'][0][0] * X[0][0]
    parameters['w1'][0][1] += 0.001 * 2 * error * parameters['w2'][0][0] * X[1][0]

    parameters['b1'][0][0] += 0.001 * 2 * error * parameters['w2'][0][0]

    parameters['w1'][1][0] += 0.001 * 2 * error * parameters['w2'][1][0] * X[0][0]
    parameters['w1'][1][1] += 0.001 * 2 * error * parameters['w2'][1][0] * X[1][0]

    parameters['b1'][1][0] += 0.001 * 2 * error * parameters['w2'][1][0]

In [160]:
# epochs implementation

parameters = initialize_parameter([2,2,1])
epochs = 5

for i in range(epochs):

  Loss = []

  for j in range(df.shape[0]):

    X = df[['cgpa', 'profile_score']].values[j].reshape(2,1) # Shape(no of features, no. of training example)
    y = df[['lpa']].values[j][0]

    # Parameter initialization


    y_hat,A1 = L_layer_forward(X,parameters)
    y_hat = y_hat[0][0]

    update_parameters(parameters,y,y_hat,A1,X)

    Loss.append((y-y_hat)**2)

  print('Epoch - ',i+1,'Loss - ',np.array(Loss).mean())

parameters

Epoch -  1 Loss -  26.28249792398698
Epoch -  2 Loss -  19.438253848220803
Epoch -  3 Loss -  10.13987443582752
Epoch -  4 Loss -  3.385561305106485
Epoch -  5 Loss -  1.3198454128484565


{'w1': array([[0.273603  , 0.3993222 ],
        [0.28787155, 0.42586102]]),
 'b1': array([[0.02885522],
        [0.03133223]]),
 'w2': array([[0.42574893],
        [0.50219328]]),
 'b2': array([[0.11841278]])}

## Same with tensorflow keras

In [1]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense

In [3]:
model = Sequential()
model.add(Dense(2,activation='linear',input_dim = 2))
model.add(Dense(1,activation='linear'))

g:\Deep learning\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [5]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 2)              │             6 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9 (36.00 B)

 Trainable params: 9 (36.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
model.get_weights()

[array([[ 0.5785887 , -1.1527883 ],
        [-0.86812353, -1.0333998 ]], dtype=float32),
 array([0., 0.], dtype=float32),
 array([[-0.68767506],
        [-0.16727614]], dtype=float32),
 array([0.], dtype=float32)]

In [7]:
optimizer = keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss= 'mean_squared_error',optimizer=optimizer)

In [12]:
model.fit(df.iloc[:,0:-1].values,df['lpa'].values, epochs=75,verbose=1,batch_size=1)

Epoch 1/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.5045  
Epoch 2/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.3333 
Epoch 3/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2365 
Epoch 4/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.1598
Epoch 5/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1001     
Epoch 6/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0910 
Epoch 7/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0769 
Epoch 8/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0774 
Epoch 9/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0815 
Epoch 10/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0798 
Epoch 11/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0798 
Epoch 12/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0775
Epoch 13/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0752 
Epoch 14/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0729 
Epoch 15/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0712 
Epoch 16/75
4/4 ━━━━━━━━━━━━━

In [13]:
model.get_weights()

[array([[ 0.5652286, -1.165983 ],
        [-0.8382015, -1.0035082]], dtype=float32),
 array([0.01071762, 0.01077304], dtype=float32),
 array([[-0.63648707],
        [-0.15279761]], dtype=float32),
 array([-0.01049841], dtype=float32)]